In [2]:
# set the matplotlib backend so figures can be saved in the background
import matplotlib
matplotlib.use("Agg")
# import the necessary packages
import torch 
import torch.nn as nn
import torch.nn as CrossEntropyLoss
from sklearn.model_selection import train_test_split
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
# from pyimagesearch.lenet import LeNet
from imutils import paths
import matplotlib.pyplot as plt
import numpy as np
import argparse
import random
import cv2
import os

In [11]:
import os

print(os.getcwd())

c:\Users\andre


In [3]:
args = {
    "dataset": r"C:\Users\andre\OneDrive\Skrivebord\Data Science\Deep Learning\Deep-Learning-\catdog_data\catdog_data"
}

In [4]:
# initialize the data and labels
print("[INFO] loading images...")
data = []
labels = []
# grab the image paths and randomly shuffle them
imagePaths = sorted(list(paths.list_images(args["dataset"])))
random.seed(42)
random.shuffle(imagePaths)

[INFO] loading images...


In [5]:
# loop over the input images
for imagePath in imagePaths:
	# load the image, pre-process it, and store it in the data list
	image = cv2.imread(imagePath)
	image = cv2.resize(image, (28, 28))
	image = torch.tensor(image, dtype=torch.float32)
	data.append(image)
	# extract the class label from the image path and update the
	# labels list
	label = imagePath.split(os.path.sep)[-2]
	label = 1 if label == "dogs" else 0
	labels.append(label)

In [15]:
for imagePath, label in zip(imagePaths[:10], labels[:10]):
    print(f"{imagePath} → {label}")

C:\Users\andre\OneDrive\Skrivebord\Data Science\Deep Learning\Deep-Learning-\catdog_data\catdog_data\dogs\dog.641.jpg → 1
C:\Users\andre\OneDrive\Skrivebord\Data Science\Deep Learning\Deep-Learning-\catdog_data\catdog_data\dogs\dog.284.jpg → 1
C:\Users\andre\OneDrive\Skrivebord\Data Science\Deep Learning\Deep-Learning-\catdog_data\catdog_data\dogs\dog.295.jpg → 1
C:\Users\andre\OneDrive\Skrivebord\Data Science\Deep Learning\Deep-Learning-\catdog_data\catdog_data\dogs\dog.608.jpg → 1
C:\Users\andre\OneDrive\Skrivebord\Data Science\Deep Learning\Deep-Learning-\catdog_data\catdog_data\dogs\dog.744.jpg → 1
C:\Users\andre\OneDrive\Skrivebord\Data Science\Deep Learning\Deep-Learning-\catdog_data\catdog_data\dogs\dog.979.jpg → 1
C:\Users\andre\OneDrive\Skrivebord\Data Science\Deep Learning\Deep-Learning-\catdog_data\catdog_data\cats\cat.495.jpg → 0
C:\Users\andre\OneDrive\Skrivebord\Data Science\Deep Learning\Deep-Learning-\catdog_data\catdog_data\cats\cat.101.jpg → 0
C:\Users\andre\OneDrive\

In [6]:
# scale the raw pixel intensities to the range [0, 1]
data = np.array(data, dtype="float") / 255.0
labels = np.array(labels)

# one_hot_pytorch = F.one_hot(torch.tensor(labels), num_classes= 2)

# partition the data into training and testing splits using 75% of
# the data for training and the remaining 25% for testing
(trainX, testX, trainY, testY) = train_test_split(data,
	labels, test_size=0.25, random_state=42)

trainX = torch.tensor(trainX, dtype=torch.float32)
testX = torch.tensor(testX, dtype=torch.float32)

trainY = torch.tensor(trainY, dtype=torch.long)
testY = torch.tensor(testY, dtype=torch.long)

trainX = trainX.reshape(trainX.shape[0], -1)
testX = testX.reshape(testX.shape[0], -1)


In [9]:
trainY.shape

torch.Size([1800])

In [23]:
model = nn.Sequential(
  nn.Linear(2352,128),
  nn.Linear(128,64),
  nn.Linear(64,32),
  nn.Linear(32,2)
)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001)

output = model(trainX)
# print(output)
probs = torch.softmax(output, dim=1)
print(probs)
loss = criterion(output, trainY)
print(loss)

tensor([[0.4784, 0.5216],
        [0.4780, 0.5220],
        [0.4705, 0.5295],
        ...,
        [0.4646, 0.5354],
        [0.4672, 0.5328],
        [0.4750, 0.5250]], grad_fn=<SoftmaxBackward0>)
tensor(0.6949, grad_fn=<NllLossBackward0>)


In [ ]:
loss.backward()
optimizer.step()


tensor([[0.4789, 0.5211],
        [0.4784, 0.5216],
        [0.4709, 0.5291],
        ...,
        [0.4650, 0.5350],
        [0.4675, 0.5325],
        [0.4754, 0.5246]], grad_fn=<SoftmaxBackward0>)
tensor(0.6948, grad_fn=<NllLossBackward0>)


In [6]:
data = np.array(data, dtype="float") / 255.0 #Normalisere værdier mellem 0 og 1 
labels = np.array(labels) #Laver outcome til array så det kan bruges i split og tensor

(trainX, testX, trainY, testY) = train_test_split(data,
	labels, test_size=0.25, random_state=42) # Splitter data

trainX = trainX.reshape(trainX.shape[0], -1) #'Flattens' tranings data da det er billeder 
testX = testX.reshape(testX.shape[0], -1)

train_dataset = TensorDataset(
    torch.tensor(trainX, dtype=torch.float32), #Laver data om til tensors
    torch.tensor(trainY, dtype=torch.long)
)

test_dataset = TensorDataset(
    torch.tensor(testX, dtype=torch.float32),
    torch.tensor(testY, dtype=torch.long)
)


input_sample, label_sample = train_dataset[0] #Tjekker om det ser fornuftigt ud
print('train input sample:', input_sample)
print('train label sample:', label_sample)
input_sample, label_sample = test_dataset[0]
print('test input sample:', input_sample)
print('test label sample:', label_sample)

train input sample: tensor([0.1882, 0.2000, 0.2431,  ..., 0.7725, 0.7804, 0.8118])
train label sample: tensor(1)
test input sample: tensor([0.2667, 0.2863, 0.2824,  ..., 0.1961, 0.2549, 0.3294])
test label sample: tensor(0)


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True) #Laver data om til dataloader klasse som er nem at arbejde med i loops når der bruges torch
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

model = nn.Sequential(
  nn.Linear(2352,128),
  nn.ReLU(),
  nn.Linear(128,64),
  nn.ReLU(),
  nn.Linear(64,32),
  nn.ReLU(),
  nn.Dropout(p=0.5),
  nn.Linear(32,2)
)
model.train()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.95, weight_decay=0.0001) # weight_decay mindsker weights ved penalty 


num_epochs = 30

In [9]:
for epochs in range(num_epochs): # Trænings model 
        epoch_loss = 0
        train_correct = 0
        train_total = 0
        for trainX, trainY in train_loader:
                optimizer.zero_grad()
                pred = model(trainX)

                predicted = torch.argmax(pred, dim=1) #argmax vælger største værdi i (prediction) tensor 
                train_correct += (predicted == trainY).sum().item() #Summere rigtige predictions 
                train_total += trainY.size(0) #Summere antal batches / samlet mængde af predictions 

                loss = criterion(pred, trainY)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
        print("Train Accuracy", train_correct / train_total)
        print("Test Loss", epoch_loss / len(train_loader))
    
        


Train Accuracy 0.4922222222222222
Test Loss 0.6935549127428156
Train Accuracy 0.5238888888888888
Test Loss 0.6927370240813807
Train Accuracy 0.5033333333333333
Test Loss 0.6931803383325276
Train Accuracy 0.5038888888888889
Test Loss 0.6938658858600416
Train Accuracy 0.51
Test Loss 0.6926822861035665
Train Accuracy 0.5177777777777778
Test Loss 0.6923736312933135
Train Accuracy 0.5216666666666666
Test Loss 0.692094810176314
Train Accuracy 0.5316666666666666
Test Loss 0.6914886654469005
Train Accuracy 0.5211111111111111
Test Loss 0.6919329469664055
Train Accuracy 0.5322222222222223
Test Loss 0.6912945257989984
Train Accuracy 0.5305555555555556
Test Loss 0.6911085153880873
Train Accuracy 0.5438888888888889
Test Loss 0.6907940569676851
Train Accuracy 0.5466666666666666
Test Loss 0.6902364293734232
Train Accuracy 0.5527777777777778
Test Loss 0.6900057844948351
Train Accuracy 0.5383333333333333
Test Loss 0.6893965392781977
Train Accuracy 0.5333333333333333
Test Loss 0.6901239365862127
Train A

In [10]:
model.eval()
test_correct = 0
test_total = 0
with torch.no_grad(): #Evaluerings model 
    for testX, testY in test_loader:
        pred = model(testX)
        predicted = torch.argmax(pred, dim=1)
        test_correct += (predicted == testY).sum().item()
        test_total += testY.size(0)
    print("Train Accuracy", test_correct / test_total)

        


Train Accuracy 0.5966666666666667
